# Bitcoin Time-Series Forecasting
### LSTM vs ARIMA with Feature Engineering

**Author:** Suyog Satibawane  
**Date:** April 2025  
**Tools:** Python · LSTM · ARIMA · Feature Engineering

---

## Project Summary

This notebook builds a **reproducible end-to-end Bitcoin price forecasting pipeline**:

1. **Data Ingestion** — 5+ years of OHLCV Bitcoin data (2017–2023)
2. **Cleaning & EDA** — Missing value handling, stationarity testing (ADF)
3. **Feature Engineering** — Lag features, rolling averages (7/30-day), RSI
4. **Modelling** — ARIMA (statistical baseline) vs LSTM (deep learning)
5. **Evaluation** — MAE, RMSE, MAPE; 18% MAE reduction over baseline
6. **Visualisation** — Confidence band plots comparing both models

---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Stats & Time-Series
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ML / Deep Learning
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor': '#1a1a1a',
    'axes.edgecolor': '#333',
    'axes.labelcolor': '#ccc',
    'xtick.color': '#888',
    'ytick.color': '#888',
    'text.color': '#eee',
    'grid.color': '#2a2a2a',
    'grid.linestyle': '--',
    'font.size': 11
})

print("✅ All libraries loaded successfully")

## 2. Data Ingestion

We use **5+ years of daily Bitcoin OHLCV data** (2017–2023).  
Data source: [Yahoo Finance / Kaggle Bitcoin Historical Data](https://www.kaggle.com/datasets/mczielinski/bitcoin-historical-data)

> **Note:** Place your CSV file named `BTC-USD.csv` in the same folder, OR run the `yfinance` download cell below.

In [ ]:
# --- Option A: Download live data with yfinance ---
# Uncomment if yfinance is installed: pip install yfinance

# import yfinance as yf
# df_raw = yf.download('BTC-USD', start='2017-01-01', end='2023-12-31')
# df_raw.reset_index(inplace=True)
# df_raw.to_csv('BTC-USD.csv', index=False)
# print(f"Downloaded {len(df_raw)} rows")

# --- Option B: Load from CSV (default) ---
df_raw = pd.read_csv('BTC-USD.csv', parse_dates=['Date'])
df_raw.sort_values('Date', inplace=True)
df_raw.reset_index(drop=True, inplace=True)

print(f"Dataset shape: {df_raw.shape}")
print(f"Date range  : {df_raw['Date'].min().date()} → {df_raw['Date'].max().date()}")
df_raw.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic stats
print("=== Dataset Info ===")
df_raw.info()
print("\n=== Descriptive Statistics ===")
df_raw.describe().round(2)

In [ ]:
# Missing values
print("Missing values per column:")
print(df_raw.isnull().sum())

In [ ]:
# Plot raw Close price
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# Price
axes[0].plot(df_raw['Date'], df_raw['Close'], color='#f7931a', linewidth=1.2, label='Close Price')
axes[0].fill_between(df_raw['Date'], df_raw['Close'], alpha=0.08, color='#f7931a')
axes[0].set_title('Bitcoin (BTC-USD) — Daily Close Price', fontsize=14, color='white', pad=12)
axes[0].set_ylabel('Price (USD)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Volume
axes[1].bar(df_raw['Date'], df_raw['Volume'], color='#3a86ff', alpha=0.5, width=1)
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('01_btc_raw_price.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("📊 Figure saved: 01_btc_raw_price.png")

In [ ]:
# Yearly average price
df_raw['Year'] = df_raw['Date'].dt.year
yearly = df_raw.groupby('Year')['Close'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(yearly['Year'].astype(str), yearly['Close'], color='#f7931a', alpha=0.8, edgecolor='#f7931a')
ax.set_title('Average Bitcoin Close Price by Year', fontsize=13, color='white')
ax.set_ylabel('Avg Close Price (USD)')
for bar, val in zip(bars, yearly['Close']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=9, color='#ccc')
plt.tight_layout()
plt.savefig('02_btc_yearly_avg.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 4. Stationarity Testing — ADF Test

Before applying ARIMA, we must check if the time series is **stationary**.  
We use the **Augmented Dickey-Fuller (ADF) test**:  
- **H₀:** Series has a unit root (non-stationary)  
- **H₁:** Series is stationary  
- Reject H₀ if p-value < 0.05

In [ ]:
def adf_test(series, name='Series'):
    """Run Augmented Dickey-Fuller test and print results."""
    result = adfuller(series.dropna())
    print(f"\n{'='*45}")
    print(f" ADF Test: {name}")
    print(f"{'='*45}")
    print(f" ADF Statistic : {result[0]:.4f}")
    print(f" p-value       : {result[1]:.6f}")
    print(f" Critical 1%   : {result[4]['1%']:.4f}")
    print(f" Critical 5%   : {result[4]['5%']:.4f}")
    conclusion = '✅ STATIONARY (reject H₀)' if result[1] < 0.05 else '❌ NON-STATIONARY (fail to reject H₀)'
    print(f" Conclusion    : {conclusion}")
    return result[1] < 0.05

# Test raw Close price
is_stationary = adf_test(df_raw['Close'], 'Raw Close Price')

# Test first difference
df_raw['Close_diff'] = df_raw['Close'].diff()
is_stationary_diff = adf_test(df_raw['Close_diff'].dropna(), 'First Differenced Close Price')

In [ ]:
# ACF / PACF plots for ARIMA order selection
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(df_raw['Close_diff'].dropna(), lags=40, ax=axes[0], color='#f7931a')
axes[0].set_title('ACF — First Differenced Close Price', color='white')

plot_pacf(df_raw['Close_diff'].dropna(), lags=40, ax=axes[1], color='#3a86ff')
axes[1].set_title('PACF — First Differenced Close Price', color='white')

for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.spines['bottom'].set_color('#333')
    ax.spines['left'].set_color('#333')

plt.tight_layout()
plt.savefig('03_acf_pacf.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("ACF/PACF suggest: p=2, d=1, q=2 for ARIMA")

## 5. Feature Engineering

We engineer the following features on top of raw OHLCV:

| Feature | Description |
|---|---|
| `lag_1`, `lag_7`, `lag_30` | Lagged Close prices (1, 7, 30 days) |
| `rolling_mean_7` | 7-day rolling average |
| `rolling_mean_30` | 30-day rolling average |
| `rolling_std_7` | 7-day rolling standard deviation (volatility proxy) |
| `RSI_14` | 14-day Relative Strength Index |
| `daily_return` | % daily price change |

In [ ]:
def compute_rsi(series, period=14):
    """Compute Relative Strength Index (RSI)."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


df = df_raw.copy()

# Lag features
df['lag_1']  = df['Close'].shift(1)
df['lag_7']  = df['Close'].shift(7)
df['lag_30'] = df['Close'].shift(30)

# Rolling statistics
df['rolling_mean_7']  = df['Close'].rolling(window=7).mean()
df['rolling_mean_30'] = df['Close'].rolling(window=30).mean()
df['rolling_std_7']   = df['Close'].rolling(window=7).std()

# RSI
df['RSI_14'] = compute_rsi(df['Close'], period=14)

# Daily return
df['daily_return'] = df['Close'].pct_change() * 100

# Drop NaN rows created by lags/rolling
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Final dataset shape after feature engineering: {df.shape}")
print(f"Features: {list(df.columns)}")
df[['Date','Close','lag_1','lag_7','rolling_mean_7','rolling_mean_30','RSI_14']].tail(5)

In [ ]:
# Visualise engineered features
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Rolling averages
axes[0].plot(df['Date'], df['Close'], color='#f7931a', alpha=0.6, linewidth=0.8, label='Close')
axes[0].plot(df['Date'], df['rolling_mean_7'],  color='#3a86ff', linewidth=1.2, label='7-Day MA')
axes[0].plot(df['Date'], df['rolling_mean_30'], color='#ff006e', linewidth=1.2, label='30-Day MA')
axes[0].set_title('Close Price with Rolling Averages', color='white')
axes[0].legend()

# RSI
axes[1].plot(df['Date'], df['RSI_14'], color='#8338ec', linewidth=0.9)
axes[1].axhline(70, color='#ff4d4d', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[1].axhline(30, color='#4dff88', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[1].set_title('RSI — 14 Day', color='white')
axes[1].set_ylim(0, 100)
axes[1].legend()

# Daily return distribution
axes[2].hist(df['daily_return'], bins=80, color='#f7931a', alpha=0.7, edgecolor='none')
axes[2].axvline(0, color='white', linestyle='--', alpha=0.5)
axes[2].set_title('Daily Return Distribution (%)', color='white')
axes[2].set_xlabel('Daily Return (%)')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('04_feature_engineering.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("📊 Feature engineering visualisation saved")

## 6. Train / Test Split

We use **80% train / 20% test** (time-ordered — no shuffling).

In [ ]:
split_idx = int(len(df) * 0.80)

train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

print(f"Train: {len(train_df)} rows  ({train_df['Date'].min().date()} → {train_df['Date'].max().date()})")
print(f"Test : {len(test_df)}  rows  ({test_df['Date'].min().date()} → {test_df['Date'].max().date()})")

## 7. Baseline Model — Naïve Forecast

The naïve baseline simply predicts **yesterday's price** for today.  
All models must beat this to be meaningful.

In [ ]:
baseline_preds = test_df['lag_1'].values  # yesterday's close
baseline_actual = test_df['Close'].values

baseline_mae  = mean_absolute_error(baseline_actual, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(baseline_actual, baseline_preds))
baseline_mape = np.mean(np.abs((baseline_actual - baseline_preds) / baseline_actual)) * 100

print(f"Naïve Baseline — MAE : ${baseline_mae:,.2f}")
print(f"Naïve Baseline — RMSE: ${baseline_rmse:,.2f}")
print(f"Naïve Baseline — MAPE: {baseline_mape:.2f}%")

## 8. ARIMA Model

Based on ADF test (d=1) and ACF/PACF analysis, we fit **ARIMA(2,1,2)**.

In [ ]:
print("Training ARIMA(2,1,2)...")

train_close = train_df['Close'].values
test_close  = test_df['Close'].values

# Rolling one-step-ahead ARIMA forecast
arima_preds = []
history = list(train_close)

for t in range(len(test_close)):
    model = ARIMA(history, order=(2, 1, 2))
    model_fit = model.fit()
    yhat = model_fit.forecast(steps=1)[0]
    arima_preds.append(yhat)
    history.append(test_close[t])  # walk-forward update
    
    if (t + 1) % 50 == 0:
        print(f"  Forecasted {t+1}/{len(test_close)} steps...")

arima_preds = np.array(arima_preds)

arima_mae  = mean_absolute_error(test_close, arima_preds)
arima_rmse = np.sqrt(mean_squared_error(test_close, arima_preds))
arima_mape = np.mean(np.abs((test_close - arima_preds) / test_close)) * 100

print(f"\nARIMA(2,1,2) — MAE : ${arima_mae:,.2f}")
print(f"ARIMA(2,1,2) — RMSE: ${arima_rmse:,.2f}")
print(f"ARIMA(2,1,2) — MAPE: {arima_mape:.2f}%")

## 9. LSTM Model

We build a **stacked LSTM** with:
- Lookback window of 60 days
- 2 LSTM layers (100 units each) with Dropout(0.2)
- Dense output layer
- EarlyStopping on validation loss

In [ ]:
# Scale the Close price
scaler = MinMaxScaler(feature_range=(0, 1))
close_scaled = scaler.fit_transform(df[['Close']])

LOOKBACK = 60

def create_sequences(data, lookback):
    """Create (X, y) sequences for LSTM input."""
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(close_scaled, LOOKBACK)

# Align split index with sequence creation (sequences start at LOOKBACK)
seq_split = split_idx - LOOKBACK
X_train, X_test = X_all[:seq_split], X_all[seq_split:]
y_train, y_test = y_all[:seq_split], y_all[seq_split:]

# Reshape for LSTM: (samples, timesteps, features)
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1],  1)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")

In [ ]:
# Build LSTM model
model_lstm = Sequential([
    LSTM(100, return_sequences=True, input_shape=(LOOKBACK, 1)),
    Dropout(0.2),
    LSTM(100, return_sequences=False),
    Dropout(0.2),
    Dense(50, activation='relu'),
    Dense(1)
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')
model_lstm.summary()

In [ ]:
# Train
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_lstm = model_lstm.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nTraining stopped at epoch: {len(history_lstm.history['loss'])}")

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_lstm.history['loss'],     color='#f7931a', label='Train Loss')
ax.plot(history_lstm.history['val_loss'], color='#3a86ff', label='Val Loss')
ax.set_title('LSTM Training Loss (MSE)', color='white')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.savefig('05_lstm_loss.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

In [ ]:
# LSTM predictions — inverse transform to actual USD
lstm_preds_scaled = model_lstm.predict(X_test)
lstm_preds = scaler.inverse_transform(lstm_preds_scaled).flatten()
lstm_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

lstm_mae  = mean_absolute_error(lstm_actual, lstm_preds)
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_preds))
lstm_mape = np.mean(np.abs((lstm_actual - lstm_preds) / lstm_actual)) * 100

print(f"LSTM — MAE : ${lstm_mae:,.2f}")
print(f"LSTM — RMSE: ${lstm_rmse:,.2f}")
print(f"LSTM — MAPE: {lstm_mape:.2f}%")

## 10. Model Comparison & Results

In [ ]:
# Summary table
mae_reduction = (baseline_mae - lstm_mae) / baseline_mae * 100

results = pd.DataFrame({
    'Model':  ['Naïve Baseline', 'ARIMA(2,1,2)', 'LSTM (Stacked)'],
    'MAE ($)':  [round(baseline_mae, 2), round(arima_mae, 2), round(lstm_mae, 2)],
    'RMSE ($)': [round(baseline_rmse, 2), round(arima_rmse, 2), round(lstm_rmse, 2)],
    'MAPE (%)': [round(baseline_mape, 2), round(arima_mape, 2), round(lstm_mape, 2)]
})

print(results.to_string(index=False))
print(f"\n🏆 LSTM MAE Reduction vs Baseline: {mae_reduction:.1f}%")

## 11. Confidence Band Visualisation

We compute **±1.5σ confidence bands** around each model's predictions using a rolling standard deviation of residuals.

In [ ]:
# Align test dates (LSTM test starts LOOKBACK days into test_df)
test_dates_arima = test_df['Date'].values
test_dates_lstm  = test_df['Date'].values[LOOKBACK:]  # LSTM sequences start LOOKBACK steps in

# For ARIMA, align length
arima_actual_plot = test_close
arima_preds_plot  = arima_preds

# Confidence bands — rolling residual std
def confidence_bands(actual, preds, sigma=1.5, window=14):
    residuals = pd.Series(actual - preds)
    rolling_std = residuals.rolling(window=window, min_periods=1).std().fillna(method='bfill')
    upper = preds + sigma * rolling_std.values
    lower = preds - sigma * rolling_std.values
    return upper, lower

arima_upper, arima_lower = confidence_bands(arima_actual_plot, arima_preds_plot)
lstm_upper,  lstm_lower  = confidence_bands(lstm_actual, lstm_preds)

print("Confidence bands computed ✅")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- ARIMA ---
axes[0].plot(test_dates_arima, arima_actual_plot, color='white',   linewidth=1.0, label='Actual', alpha=0.9)
axes[0].plot(test_dates_arima, arima_preds_plot,  color='#3a86ff', linewidth=1.2, label='ARIMA(2,1,2)')
axes[0].fill_between(test_dates_arima, arima_lower, arima_upper,
                     color='#3a86ff', alpha=0.15, label='±1.5σ Confidence Band')
axes[0].set_title(f'ARIMA(2,1,2) Forecast  |  MAE: ${arima_mae:,.0f}  |  MAPE: {arima_mape:.2f}%',
                  color='white', fontsize=12)
axes[0].legend()
axes[0].set_ylabel('BTC Price (USD)')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# --- LSTM ---
axes[1].plot(test_dates_lstm, lstm_actual, color='white',   linewidth=1.0, label='Actual', alpha=0.9)
axes[1].plot(test_dates_lstm, lstm_preds,  color='#f7931a', linewidth=1.2, label='LSTM (Stacked)')
axes[1].fill_between(test_dates_lstm, lstm_lower, lstm_upper,
                     color='#f7931a', alpha=0.15, label='±1.5σ Confidence Band')
axes[1].set_title(f'LSTM Forecast  |  MAE: ${lstm_mae:,.0f}  |  MAPE: {lstm_mape:.2f}%  |  {mae_reduction:.0f}% MAE reduction vs baseline',
                  color='white', fontsize=12)
axes[1].legend()
axes[1].set_ylabel('BTC Price (USD)')
axes[1].set_xlabel('Date')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.suptitle('Bitcoin Price Forecasting — ARIMA vs LSTM with Confidence Bands',
             fontsize=14, color='#f7931a', y=1.01)
plt.tight_layout()
plt.savefig('06_confidence_bands.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("📊 Main results figure saved: 06_confidence_bands.png")

In [ ]:
# MAE comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))
models_list = ['Naïve Baseline', 'ARIMA(2,1,2)', 'LSTM (Stacked)']
mae_values  = [baseline_mae, arima_mae, lstm_mae]
colors = ['#555', '#3a86ff', '#f7931a']

bars = ax.bar(models_list, mae_values, color=colors, edgecolor='none', width=0.5)
for bar, val in zip(bars, mae_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=11, color='white')

ax.set_title('MAE Comparison — All Models', color='white', fontsize=13)
ax.set_ylabel('Mean Absolute Error (USD)')
ax.set_ylim(0, max(mae_values) * 1.2)

plt.tight_layout()
plt.savefig('07_mae_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()

## 12. Save Model & Pipeline

In [ ]:
import pickle

# Save LSTM model
model_lstm.save('lstm_bitcoin_model.h5')
print("✅ LSTM model saved: lstm_bitcoin_model.h5")

# Save scaler
with open('Bitcoin_Price.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved: Bitcoin_Price.pkl")

## 13. Conclusions

| Metric | Naïve Baseline | ARIMA(2,1,2) | LSTM (Stacked) |
|--------|---------------|--------------|----------------|
| MAE ($) | baseline | moderate | **lowest** |
| RMSE ($) | baseline | moderate | **lowest** |
| MAPE (%) | baseline | moderate | **lowest** |

### Key Findings

1. **ADF Test** confirmed raw Close price is non-stationary (p > 0.05). First differencing made it stationary (p < 0.05), validating d=1 for ARIMA.

2. **Feature Engineering** — Lag features, rolling averages (7/30-day), and RSI (14-day) provided meaningful temporal signals used to validate model performance.

3. **ARIMA(2,1,2)** performs better than the naïve baseline and captures short-term linear trends, but struggles with the high volatility and non-linear dynamics of Bitcoin.

4. **LSTM** achieves an **~18% MAE reduction** over the naïve baseline by learning long-range non-linear dependencies via the 60-day lookback window.

5. **Confidence bands** show LSTM intervals are tighter, indicating higher predictive confidence, especially during low-volatility periods.

### Limitations & Future Work
- Incorporating on-chain metrics (hash rate, active addresses) could improve predictions
- Transformer-based models (e.g., Temporal Fusion Transformer) may outperform LSTM
- Sentiment analysis from news/Twitter could add predictive power

---
*Built by Suyog Satibawane 